<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# **Simple GAN on CelebA**
**Architecture:** DCGAN (Radford et al., 2015)
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab

- Aissa Berenice González Fosado
- Daniela de la Torre Gallo
- Clara Paola Aguilar Casillas


This is the **baseline GAN** before AttGAN. Comparing both shows the advantage of
conditional attribute-guided generation.

| | Simple GAN (this notebook) | AttGAN |
|---|---|---|
| Conditioning | None — random noise only | Attribute vector {-1, +1} |
| Control over output | None | Edit 13 specific facial attributes |
| Loss | BCE adversarial only | Adv + Classification + Reconstruction |
| Output resolution | 64x64 | 128x128 |


## *Clone repo & install*

In [1]:
!git clone https://github.com/AissaBerenice/GAN_Project1-DL.git 2>/dev/null || echo 'Repo already cloned'
%cd GAN_Project1-DL
!pip install -q -r requirements.txt

print('Setup complete')

/content/GAN_Project1-DL
Setup complete


## *Verify GPU & config*

In [2]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change Runtime Type > T4 GPU'
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from config import Config

cfg = Config()
cfg.EXPERIMENT_NAME = 'simple_gan'
cfg.__init__()   # creates results/simple_gan/ and checkpoints/simple_gan/

device = torch.device('cuda')

# Reduce for a quick smoke test:
# cfg.N_EPOCHS = 5

print(f'Experiment  : {cfg.EXPERIMENT_NAME}')
print(f'Epochs      : {cfg.N_EPOCHS}')
print(f'Batch size  : {cfg.BATCH_SIZE}')
print(f'Results dir : {cfg.RESULTS_DIR}')

Python  : 3.12.13
PyTorch : 2.10.0+cu128
GPU     : NVIDIA H100 80GB HBM3
VRAM    : 85.0 GB
Experiment  : simple_gan
Epochs      : 10
Batch size  : 32
Results dir : /content/GAN_Project1-DL/results/simple_gan



## *Load CelebA*
torchvision downloads CelebA automatically on first run (~1.4 GB).  
Images are loaded at 128×128 and downsampled to **64×64** inside the training loop.



In [25]:
#from src.dataset import get_loaders

#train_loader, test_loader = get_loaders(cfg)

#imgs, attrs = next(iter(train_loader))
#print(f'Image batch : {imgs.shape}  (loaded at 128x128, downsampled to 64x64 in training)')
#print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

This had to be run under Fallback with the Kaggle dataset due to unaviability of the torchvision one. It is at the bottom of the notebook

## *Build models*
```
noise z (100 x 1 x 1)
  |-> Generator  (5x ConvTranspose2d + BatchNorm + ReLU -> Tanh)
      fake image (3 x 64 x 64)
        |-> Discriminator  (5x Conv2d + BatchNorm + LeakyReLU -> Sigmoid)
            real/fake scalar

Loss G : BCE( D(G(z)) , 1 )            <- fool discriminator
Loss D : BCE( D(x)    , 1 )            <- real images -> 1
       + BCE( D(G(z)) , 0 )            <- fake images -> 0
```

In [10]:
from src.simple_gan import build_simple_models

LATENT_DIM = 100
gen, dis = build_simple_models(latent_dim=LATENT_DIM, device=device)

[simple_gan] Generator=3.58M  Discriminator=2.77M



## Train
Sample grids are saved to `results/simple_gan/` every 5 epochs.  
A healthy run: both G and D losses decrease then oscillate — **neither should reach 0**.

In [11]:
from src.simple_gan import train_simple_gan

g_losses, d_losses = train_simple_gan(
    gen, dis, train_loader, cfg, device, latent_dim=LATENT_DIM
)


[simple_gan] Training 10 epochs  (experiment: simple_gan)


Epoch [  1/10]  G=4.4380  D=0.5787
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch001.png


Epoch [  2/10]  G=2.9464  D=0.6354
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch002.png


Epoch [  3/10]  G=3.0333  D=0.5668


Epoch [  4/10]  G=3.1159  D=0.5254
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch004.png


...

Epoch [  6/10]  G=3.7615  D=0.4007
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch006.png


Epoch [  7/10]  G=4.0980  D=0.3433


Epoch [  8/10]  G=4.3362  D=0.3051
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch008.png


Epoch [  9/10]  G=4.5183  D=0.2975


Epoch [ 10/10]  G=4.7502  D=0.2590
[simple_gan] Samples -> /content/GAN_Project1-DL/results/simple_gan/simple_gan_epoch010.png
[simple_gan] Training complete.


## *Loss curves*

In [12]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

[utils] Saved -> /content/GAN_Project1-DL/results/simple_gan/loss_curves.png


## *FID & DACID*
**FID** — Frechet Inception Distance. Standard GAN quality metric. Lower = better.  
**DACID** — custom metric: L2 distance between mean Inception feature vectors. Lower = better.


In [13]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics_simple_gan
    scores = compute_metrics_simple_gan(gen, test_loader, LATENT_DIM, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')


[metrics] Building Inception extractor...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 189MB/s] 


[metrics] Extracting features (512 images each)...
[metrics] FID   = 64.6355
[metrics] DACID = 4.1346
FID   : 64.6355
DACID : 4.1346


## *Save checkpoint & metrics.json*

In [14]:
import json, torch

# Save model weights
ckpt = cfg.CHECKPOINT_DIR / 'simple_gan_final.pt'
torch.save({'gen': gen.state_dict(), 'dis': dis.state_dict()}, ckpt)
print(f'Checkpoint -> {ckpt}')

# Save metrics.json so export_results.py can include this model
payload = {
    'experiment': cfg.EXPERIMENT_NAME,
    'model':      'SimpleGAN',
    'fid':        scores.get('fid'),
    'dacid':      scores.get('dacid'),
    'g_losses':   g_losses,
    'd_losses':   d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Metrics  -> {out}')

Checkpoint -> /content/GAN_Project1-DL/checkpoints/simple_gan/simple_gan_final.pt
Metrics  -> /content/GAN_Project1-DL/results/simple_gan/metrics.json


## *Download results ZIP*

In [15]:
import shutil
from google.colab import files
shutil.make_archive('simple_gan_results', 'zip', cfg.RESULTS_DIR)
files.download('simple_gan_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## *Fallback*
If the CelebA quota exceeded

Use one of these if Cell 3 raises `FileURLRetrievalError: Too many users`.


In [3]:
# 1. Go to kaggle.com > Account > Create New Token  (downloads kaggle.json)
# 2. Upload kaggle.json via the Colab Files panel (left sidebar)
# 3. Uncomment and run:

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d jessicali9530/celeba-dataset -p data/
!unzip -q data/celeba-dataset.zip -d data/

# Then re-run Cell 3
print('Uncomment above lines after uploading kaggle.json')

cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
100% 1.33G/1.33G [00:05<00:00, 262MB/s]

Uncomment above lines after uploading kaggle.json


We did have issues with torchvision this time, so we correct our functions before going back

In [4]:
!pip install -q pandas

In [5]:
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

_SPLIT_MAP = {"train": 0, "valid": 1, "test": 2}

class KaggleCelebADataset(Dataset):
    def __init__(self, data_dir, split, attr_names, img_size):
        data_dir = Path(data_dir)

        # Find the image folder — handle both possible locations
        for candidate in [
            data_dir / "img_align_celeba",
            data_dir / "celeba" / "img_align_celeba",
        ]:
            if candidate.exists():
                self._img_dir = candidate
                base = candidate.parent
                break
        else:
            raise FileNotFoundError(
                "Cannot find img_align_celeba/ folder inside data/. "
                "Run the unzip cell first."
            )

        # Find CSV files
        for csv_name in ["list_eval_partition.csv", "list_eval_partition.txt"]:
            p = base / csv_name
            if p.exists():
                part_df = pd.read_csv(p)
                break
        else:
            raise FileNotFoundError("Cannot find list_eval_partition.csv")

        for csv_name in ["list_attr_celeba.csv", "list_attr_celeba.txt"]:
            p = base / csv_name
            if p.exists():
                attr_df = pd.read_csv(p)
                break
        else:
            raise FileNotFoundError("Cannot find list_attr_celeba.csv")

        # Standardise column names
        part_df.columns = part_df.columns.str.strip()
        attr_df.columns  = attr_df.columns.str.strip()

        # Rename first column to 'image_id' if needed
        if part_df.columns[0] != 'image_id':
            part_df = part_df.rename(columns={part_df.columns[0]: 'image_id'})
        if attr_df.columns[0] != 'image_id':
            attr_df = attr_df.rename(columns={attr_df.columns[0]: 'image_id'})

        # Rename partition column if needed
        pcol = [c for c in part_df.columns if c != 'image_id'][0]
        part_df = part_df.rename(columns={pcol: 'partition'})

        merged   = part_df.merge(attr_df, on='image_id')
        split_df = merged[merged['partition'] == _SPLIT_MAP[split]].reset_index(drop=True)

        self._filenames  = split_df['image_id'].tolist()
        # Kaggle attrs are already {-1,+1}; torchvision uses {0,1} — handle both
        raw = split_df[attr_names].values.astype('float32')
        # If all values are 0 or 1, convert to {-1,+1}
        if raw.max() <= 1.0 and raw.min() >= 0.0:
            raw = raw * 2 - 1
        self._attrs = raw

        self._transform = T.Compose([
            T.CenterCrop(178),
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self): return len(self._filenames)

    def __getitem__(self, i):
        img = Image.open(self._img_dir / self._filenames[i]).convert('RGB')
        return self._transform(img), torch.from_numpy(self._attrs[i])


def get_loaders(cfg):
    kw = dict(data_dir=cfg.DATA_DIR, attr_names=cfg.ATTRS, img_size=cfg.IMG_SIZE)
    train_ds = KaggleCelebADataset(split='train', **kw)
    test_ds  = KaggleCelebADataset(split='test',  **kw)
    lkw = dict(num_workers=2, pin_memory=True, drop_last=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  **lkw)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, **lkw)
    print(f"[dataset] train={len(train_ds):,}  test={len(test_ds):,}")
    return train_loader, test_loader

# Make this the active loader for this session
import src.dataset as _ds_module
import src.simple_gan as _sg_module
_ds_module.get_loaders = get_loaders
print("Dataset loader patched for Kaggle files.")

Dataset loader patched for Kaggle files.


In [6]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

[dataset] train=162,770  test=19,962


Fixing the path from kaggle, it is different

In [7]:
from pathlib import Path

# Point the loader to the correct nested folder
correct_path = Path('data/img_align_celeba/img_align_celeba')
train_loader.dataset._img_dir = correct_path
test_loader.dataset._img_dir  = correct_path

print(f"img_dir set to: {correct_path}")
print(f"Exists: {correct_path.exists()}")
print(f"Sample files: {list(correct_path.iterdir())[:3]}")

img_dir set to: data/img_align_celeba/img_align_celeba
Exists: True
Sample files: [PosixPath('data/img_align_celeba/img_align_celeba/158933.jpg'), PosixPath('data/img_align_celeba/img_align_celeba/075701.jpg'), PosixPath('data/img_align_celeba/img_align_celeba/198830.jpg')]


In [8]:
imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

Image batch : torch.Size([32, 3, 128, 128])
Pixel range : [-1.00, 1.00]


Fixing Train

In [9]:
from pathlib import Path
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

_SPLIT_MAP = {"train": 0, "valid": 1, "test": 2}

class KaggleCelebADataset(Dataset):
    def __init__(self, img_dir, base_dir, split, attr_names, img_size):
        self._img_dir = Path(img_dir)

        part_df = pd.read_csv(Path(base_dir) / 'list_eval_partition.csv')
        attr_df  = pd.read_csv(Path(base_dir) / 'list_attr_celeba.csv')
        part_df.columns = part_df.columns.str.strip()
        attr_df.columns  = attr_df.columns.str.strip()

        if part_df.columns[0] != 'image_id':
            part_df = part_df.rename(columns={part_df.columns[0]: 'image_id'})
        if attr_df.columns[0] != 'image_id':
            attr_df = attr_df.rename(columns={attr_df.columns[0]: 'image_id'})

        pcol = [c for c in part_df.columns if c != 'image_id'][0]
        part_df = part_df.rename(columns={pcol: 'partition'})

        merged   = part_df.merge(attr_df, on='image_id')
        split_df = merged[merged['partition'] == _SPLIT_MAP[split]].reset_index(drop=True)

        self._filenames = split_df['image_id'].tolist()
        raw = split_df[attr_names].values.astype('float32')
        if raw.max() <= 1.0 and raw.min() >= 0.0:
            raw = raw * 2 - 1
        self._attrs = raw

        self._transform = T.Compose([
            T.CenterCrop(178),
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self): return len(self._filenames)

    def __getitem__(self, i):
        img = Image.open(self._img_dir / self._filenames[i]).convert('RGB')
        return self._transform(img), torch.from_numpy(self._attrs[i])


# Hard-coded correct paths based on your folder structure
IMG_DIR  = 'data/img_align_celeba/img_align_celeba'
BASE_DIR = 'data'

kw = dict(img_dir=IMG_DIR, base_dir=BASE_DIR,
          attr_names=cfg.ATTRS, img_size=cfg.IMG_SIZE)

train_ds = KaggleCelebADataset(split='train', **kw)
test_ds  = KaggleCelebADataset(split='test',  **kw)

lkw = dict(num_workers=2, pin_memory=True, drop_last=True)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  **lkw)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, **lkw)

print(f"train={len(train_ds):,}  test={len(test_ds):,}")

# Verify
imgs, attrs = next(iter(train_loader))
print(f"Batch: {imgs.shape}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print("Loaders ready — run the train cell now.")

train=162,770  test=19,962
Batch: torch.Size([32, 3, 128, 128])  range=[-1.00, 1.00]
Loaders ready — run the train cell now.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('data', exist_ok=True)
# !ln -sfn '/content/drive/MyDrive/YOUR_CELEBA_FOLDER' data/celeba
# # Then re-run Cell 3
print('Uncomment above and replace YOUR_CELEBA_FOLDER with your path')

In [30]:
# ── Hyperparameter verification ──────────────────────────────────────
print("=" * 52)
print(f"  EXPERIMENT      : {cfg.EXPERIMENT_NAME}")
print("=" * 52)

print(f"\n  TRAINING")
print(f"    Epochs        : {cfg.N_EPOCHS}  (~{cfg.N_EPOCHS * 9} min on T4)")
print(f"    Batch size    : {cfg.BATCH_SIZE}")
print(f"    Learning rate : {cfg.LR}")
print(f"    Beta1, Beta2  : {cfg.BETA1}, {cfg.BETA2}")

print(f"\n  DATASET")
print(f"    Image size    : {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
print(f"    Attributes    : {cfg.N_ATTRS}  {cfg.ATTRS}")

print(f"\n  ATTGAN LOSS WEIGHTS")
print(f"    λ_rec         : {cfg.LAMBDA_REC}   (reconstruction identity)")
print(f"    λ_cls_D       : {cfg.LAMBDA_CLS_D}    (discriminator classification)")
print(f"    λ_cls_G       : {cfg.LAMBDA_CLS_G}     (generator classification)")

print(f"\n  ARCHITECTURE")
print(f"    Encoder dim   : {cfg.ENC_DIM}")
print(f"    Generator dim : {cfg.DEC_DIM}")
print(f"    Discriminator : {cfg.DIS_DIM}")

print(f"\n  LOGGING & METRICS")
print(f"    Save every    : {cfg.SAVE_EVERY} epochs")
print(f"    Log every     : {cfg.LOG_EVERY_STEPS} steps")
print(f"    FID/DACID     : {'ON' if cfg.COMPUTE_METRICS else 'OFF'}  "
      f"({cfg.METRICS_N_SAMPLES} samples)")

print(f"\n  PATHS")
print(f"    Data          : {cfg.DATA_DIR}")
print(f"    Results       : {cfg.RESULTS_DIR}")
print(f"    Checkpoints   : {cfg.CHECKPOINT_DIR}")

# Credit cost estimate
mins_per_epoch_simple = 2.5
mins_per_epoch_attgan = 9
is_attgan = hasattr(cfg, 'LAMBDA_REC')
mins = cfg.N_EPOCHS * (mins_per_epoch_attgan if is_attgan else mins_per_epoch_simple)
print(f"\n  ESTIMATED RUNTIME : ~{mins:.0f} min on T4")
print("=" * 52)

# Sanity checks
assert cfg.BATCH_SIZE <= 64,   "BATCH_SIZE > 64 may cause OOM on T4"
assert cfg.N_EPOCHS   >= 5,    "N_EPOCHS < 5 won't produce meaningful results"
assert cfg.LR         <= 5e-4, "LR too high — training will be unstable"
assert cfg.IMG_SIZE   in [64, 128], "IMG_SIZE should be 64 or 128"
print("\n  All sanity checks passed.")

  EXPERIMENT      : simple_gan

  TRAINING
    Epochs        : 30  (~270 min on T4)
    Batch size    : 32
    Learning rate : 0.0002
    Beta1, Beta2  : 0.5, 0.999

  DATASET
    Image size    : 128×128
    Attributes    : 13  ['Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Bushy_Eyebrows', 'Eyeglasses', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'No_Beard', 'Pale_Skin', 'Young']

  ATTGAN LOSS WEIGHTS
    λ_rec         : 100.0   (reconstruction identity)
    λ_cls_D       : 10.0    (discriminator classification)
    λ_cls_G       : 1.0     (generator classification)

  ARCHITECTURE
    Encoder dim   : 64
    Generator dim : 64
    Discriminator : 64

  LOGGING & METRICS
    Save every    : 5 epochs
    Log every     : 100 steps
    FID/DACID     : ON  (2048 samples)

  PATHS
    Data          : /content/GAN_Project1-DL/data
    Results       : /content/GAN_Project1-DL/results/simple_gan
    Checkpoints   : /content/GAN_Project1-DL/checkpoints/simple_gan

  ESTIMATED RUNT